## 1. Mount Google Drive
Mount Drive and create a dedicated LoRA outputs root so existing non-LoRA artifacts are not overwritten.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
Path('/content/drive/MyDrive/afriqa_outputs_lora_only').mkdir(parents=True, exist_ok=True)
print('Drive mounted and LoRA output root ready.')

## 2. Clone Repository and Install Dependencies
Clone (or update) the project and install required packages for training/evaluation and LoRA.

In [ ]:
import os

if not os.path.exists('afriqa-entity-aware-qa'):
    !git clone https://github.com/OmondiKevin/afriqa-entity-aware-qa.git

%cd afriqa-entity-aware-qa
!git pull
!pip install -e .
!pip install sentence-transformers sentencepiece protobuf peft

## 3. Configure Environment and Drive-Backed Outputs
Link local `outputs/` to a dedicated Drive folder and set deterministic seed.

In [ ]:
import os
from pathlib import Path
import torch
from transformers import set_seed

set_seed(42)

# Reduce CUDA memory fragmentation during long multitask runs.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/afriqa_outputs_lora_only')
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if os.path.lexists('outputs'):
    !rm -rf outputs
!ln -s {DRIVE_OUTPUT_ROOT} outputs

for d in ['checkpoints', 'predictions', 'metrics', 'logs']:
    (DRIVE_OUTPUT_ROOT / d).mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    print('GPU detected:', GPU_NAME)
else:
    GPU_NAME = 'cpu'
    print('No GPU detected (CPU runtime).')

print('Using Drive-backed outputs at:', DRIVE_OUTPUT_ROOT)

## 4. Download Dataset
Download and subset AfriQA + MasakhaNER resources required for multitask training.

In [ ]:
!python scripts/00_download_and_subset.py --config configs/default.yaml

## 5. Preprocess Dataset
Prepare QA seq2seq files and multitask seq2seq files used by LoRA training.

In [ ]:
!python scripts/01_prepare_qa_data.py --config configs/default.yaml
!python scripts/01b_prepare_multitask_data.py --config configs/default.yaml

## 6. Confirm LoRA-Capable Configuration and Active Model
Create a dedicated LoRA config for multitask mT5 and verify relevant settings before training.

In [ ]:
import yaml
from pathlib import Path

base_cfg = Path('configs/default.yaml')
mt5_lora_cfg = Path('configs/colab_lora_mt5.yaml')

with open(base_cfg, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['model']['base'] = 'google/mt5-base'
cfg.setdefault('lora', {})['use_lora'] = True
cfg['run']['multitask_output_dir'] = 'outputs/checkpoints/multitask_mt5'
cfg['run']['multitask_pred_path'] = 'outputs/predictions/multitask_mt5_test.jsonl'

# GPU-aware stability-first profile for LoRA training.
# Note: LoRA path currently does not enable gradient checkpointing in training script,
# so very large per-device batch sizes can OOM even on A100.
gpu = (GPU_NAME or '').lower()
train_cfg = cfg.setdefault('train', {})

if 'a100' in gpu:
    train_cfg['batch_size'] = 16
    train_cfg['grad_accum'] = 4
    train_cfg['bf16'] = True
    train_cfg['fp16'] = False
elif 'l4' in gpu or 'v100' in gpu:
    train_cfg['batch_size'] = 8
    train_cfg['grad_accum'] = 4
    train_cfg['bf16'] = False
    train_cfg['fp16'] = True
elif 't4' in gpu:
    train_cfg['batch_size'] = 4
    train_cfg['grad_accum'] = 8
    train_cfg['bf16'] = False
    train_cfg['fp16'] = True
else:
    # Conservative default for unknown GPUs.
    train_cfg['batch_size'] = 4
    train_cfg['grad_accum'] = 8
    train_cfg['bf16'] = False
    train_cfg['fp16'] = True

train_cfg['num_workers'] = 4
train_cfg['logging_steps'] = 20
train_cfg['eval_steps'] = 100
train_cfg['save_steps'] = 100

with open(mt5_lora_cfg, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=False)

print('Wrote config:', mt5_lora_cfg)
print('LoRA enabled:', cfg['lora']['use_lora'])
print('Model base:', cfg['model']['base'])
print('GPU profile:', GPU_NAME)
print('batch_size:', train_cfg['batch_size'], 'grad_accum:', train_cfg['grad_accum'], 'fp16:', train_cfg['fp16'], 'bf16:', train_cfg['bf16'])
print('Note: training script auto-generates *_lora* names for mT5 when lora.use_lora=true.')

## 7. Train Multitask mT5-base with LoRA
Run sequential multitask (NER -> QA) with LoRA adapters enabled.

In [ ]:
import gc
import torch

# Clear stale allocations before long training phase.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

!python scripts/03_train_multitask_qa.py --config configs/colab_lora_mt5.yaml --sequential

# Preserve a distinct training log name for this experiment.
!cp outputs/logs/03_train_multitask_qa.log outputs/logs/03_train_multitask_mt5_lora.log

## 8. Evaluate Multitask mT5-base with LoRA
Evaluate QA-only rows and save LoRA-specific metrics.

In [ ]:
!python scripts/04_eval_predictions.py --config configs/colab_lora_mt5.yaml --pred_path outputs/predictions/multitask_mt5_lora_test.jsonl --qa_only

# Preserve a distinct eval log name for this experiment.
!cp outputs/logs/04_eval_predictions.log outputs/logs/04_eval_predictions_mt5_lora.log

## 9. Check ByT5 + LoRA Support and Configure (Conditional)
If the current code supports ByT5 with LoRA, create a separate config with distinct output paths.

In [ ]:
import re
import yaml
from pathlib import Path

train_script = Path('scripts/03_train_multitask_qa.py').read_text(encoding='utf-8')

# Conservative support check: LoRA injection exists and model selection is generic (not mT5-only).
has_peft = 'get_peft_model' in train_script and 'LoraConfig' in train_script
is_model_generic = 'model_name = model_cfg.get("base"' in train_script and 'AutoModelForSeq2SeqLM.from_pretrained(model_name)' in train_script

SUPPORTS_BYT5_LORA = bool(has_peft and is_model_generic)
print('ByT5 + LoRA support detected:', SUPPORTS_BYT5_LORA)

byt5_lora_cfg = Path('configs/colab_lora_byt5.yaml')
if SUPPORTS_BYT5_LORA:
    with open('configs/default.yaml', 'r', encoding='utf-8') as f:
        cfg_b = yaml.safe_load(f)

    cfg_b['model']['base'] = 'google/byt5-base'
    cfg_b.setdefault('lora', {})['use_lora'] = True
    cfg_b['run']['multitask_output_dir'] = 'outputs/checkpoints/multitask_byt5_lora'
    cfg_b['run']['multitask_pred_path'] = 'outputs/predictions/multitask_byt5_lora_test.jsonl'

    # ByT5 uses longer context in training script (>=1024), so keep safer per-device batch.
    train_b = cfg_b.setdefault('train', {})
    gpu = (GPU_NAME or '').lower()

    if 'a100' in gpu:
        train_b['batch_size'] = 8
        train_b['grad_accum'] = 4
        train_b['bf16'] = True
        train_b['fp16'] = False
    elif 'l4' in gpu or 'v100' in gpu:
        train_b['batch_size'] = 4
        train_b['grad_accum'] = 8
        train_b['bf16'] = False
        train_b['fp16'] = True
    elif 't4' in gpu:
        train_b['batch_size'] = 2
        train_b['grad_accum'] = 16
        train_b['bf16'] = False
        train_b['fp16'] = True
    else:
        train_b['batch_size'] = 2
        train_b['grad_accum'] = 16
        train_b['bf16'] = False
        train_b['fp16'] = True

    train_b['num_workers'] = 4
    train_b['logging_steps'] = 20
    train_b['eval_steps'] = 100
    train_b['save_steps'] = 100

    with open(byt5_lora_cfg, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg_b, f, sort_keys=False, allow_unicode=False)

    print('Wrote config:', byt5_lora_cfg)
    print('GPU profile:', GPU_NAME)
    print('ByT5 batch_size:', train_b['batch_size'], 'grad_accum:', train_b['grad_accum'], 'fp16:', train_b['fp16'], 'bf16:', train_b['bf16'])
else:
    print('Skipping ByT5 LoRA config creation (support not detected).')

## 10. Train Multitask ByT5-base with LoRA (If Supported)
Run only when support was detected in the previous step.

In [ ]:
import gc
import subprocess
import torch

if SUPPORTS_BYT5_LORA:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    subprocess.run(
        ['python', 'scripts/03_train_multitask_qa.py', '--config', 'configs/colab_lora_byt5.yaml', '--sequential'],
        check=True,
    )
    subprocess.run(
        ['cp', 'outputs/logs/03_train_multitask_qa.log', 'outputs/logs/03_train_multitask_byt5_lora.log'],
        check=True,
    )
    print('ByT5 LoRA training complete.')
else:
    print('ByT5 LoRA training skipped: support not detected.')

## 11. Evaluate Multitask ByT5-base with LoRA (If Supported)
Evaluate QA-only rows and save ByT5-LoRA-specific metrics if training ran.

In [ ]:
import subprocess

if SUPPORTS_BYT5_LORA:
    subprocess.run(
        [
            'python', 'scripts/04_eval_predictions.py',
            '--config', 'configs/colab_lora_byt5.yaml',
            '--pred_path', 'outputs/predictions/multitask_byt5_lora_test.jsonl',
            '--qa_only',
        ],
        check=True,
    )
    subprocess.run(
        ['cp', 'outputs/logs/04_eval_predictions.log', 'outputs/logs/04_eval_predictions_byt5_lora.log'],
        check=True,
    )
    print('ByT5 LoRA evaluation complete.')
else:
    print('ByT5 LoRA evaluation skipped: support not detected.')

## 12. Verify Saved Outputs and Checkpoints in Drive
List LoRA outputs to confirm predictions, metrics, logs, and checkpoints were persisted.

In [ ]:
!ls -lh outputs/checkpoints | sed -n '1,200p'
!ls -lh outputs/predictions | sed -n '1,200p'
!ls -lh outputs/metrics | sed -n '1,200p'
!ls -lh outputs/logs | sed -n '1,200p'

## 13. Display Final LoRA Metrics Summary
Print available overall metrics for LoRA experiments in JSON format.

In [ ]:
import json
from pathlib import Path

metrics_map = {
    'multitask_mt5_lora_qa_only': Path('outputs/metrics/multitask_mt5_lora_test_qa_only_metrics.json'),
    'multitask_byt5_lora_qa_only': Path('outputs/metrics/multitask_byt5_lora_test_qa_only_metrics.json'),
}

for name, path in metrics_map.items():
    print(f'\n=== {name} ===')
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        print(json.dumps(data.get('overall', {}), indent=2, ensure_ascii=False))
    else:
        print(f'Missing metrics file: {path}')